# The Agent Loop: ReAct vs. State Machines
This notebook demonstrates the evolution of the Agent Loop from fragile `while` loops to robust Directed Graphs.

## 1. The Standard ReAct Loop (Fragile)
Standard ReAct agents rely on text parsing and basic loops.

In [ ]:
import re

def fragile_react_loop(llm_output: str):
    print("--- Parsing LLM Output ---")
    print(llm_output)
    
    # Fragile Regex Parsing
    match = re.search(r'Action:\s*([\w_]+)\((.*)\)', llm_output)
    if match:
        tool_name = match.group(1)
        args = match.group(2)
        print(f"\n✅ Parsed successfully! Executing: {tool_name} with {args}")
    else:
        print("\n❌ Error: Failed to parse LLM Action format.")

# Works fine if the LLM is perfect
fragile_react_loop("Thought: I need data.\nAction: search(query='apple')")

# Fails if the LLM hallucinates spacing or forgets a colon
fragile_react_loop("Thought: I need data.\nAction search query='apple'")

--- Parsing LLM Output ---
Thought: I need data.
Action: search(query='apple')

✅ Parsed successfully! Executing: search with query='apple'
--- Parsing LLM Output ---
Thought: I need data.
Action search query='apple'

❌ Error: Failed to parse LLM Action format.


## 2. State Machines (LangGraph)
SOTA architectures replace the text-parsing `while` loop with a formal State Machine. LLMs output structured JSON tool calls instead of text.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# Define the State
class AgentState(TypedDict):
    messages: list[str]

# Define the Nodes
def agent_node(state: AgentState):
    print("🤖 Agent Node: Analyzing state and choosing tool via JSON.")
    return {"messages": ["[ToolCall: get_weather('Paris')]"]}

def tool_node(state: AgentState):
    print("🛠️ Tool Node: Executing get_weather.")
    return {"messages": ["[Observation: 72F and sunny]"]}

# Build the Graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tool", tool_node)

# Define Logic/Edges
workflow.set_entry_point("agent")
workflow.add_edge("agent", "tool")
workflow.add_edge("tool", END)

app = workflow.compile()
print("\n✅ Compiled Graph Execution:")
app.invoke({"messages": ["What is the weather in Paris?"]})


✅ Compiled Graph Execution:
🤖 Agent Node: Analyzing state and choosing tool via JSON.
🛠️ Tool Node: Executing get_weather.
